# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [3]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [4]:
feature_columns = contract["features"]

# Árboles: no necesitan escalado, solo imputación por mediana
tree_preprocessor = ColumnTransformer(
    [("num", SimpleImputer(strategy="median"), feature_columns)]
)

# Modelos basados en distancia o en gradiente: imputación + estandarización
scaled_preprocessor = ColumnTransformer(
    [("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), feature_columns)]
)

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Porque si el escalador se ajusta sobre todo el conjunto de entrenamiento antes de la validación cruzada, la media y la desviación estándar ya incorporan información de los folds de validación. Cada fold "ve" estadísticas de datos que en teoría no conoce, lo que produce fuga de información (data leakage) y estimaciones de desempeño optimistas. Dentro del `Pipeline`, `cross_validate` y `GridSearchCV` reajustan el escalador en cada fold solo con las filas de entrenamiento de ese fold y luego lo aplican a las filas de validación, que es exactamente lo que ocurrirá en producción con datos nuevos. Además, el pipeline empaqueta preprocesamiento y modelo en un solo objeto, así que se guarda y se reutiliza sin riesgo de aplicar un escalador distinto al del entrenamiento.

## 3. Modelos candidatos

In [5]:
models = {
    "DecisionTree": Pipeline([
        ("prep", clone(tree_preprocessor)),
        ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "LogisticRegression": Pipeline([
        ("prep", clone(scaled_preprocessor)),
        ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
    ]),
    "KNN": Pipeline([
        ("prep", clone(scaled_preprocessor)),
        ("clf", KNeighborsClassifier()),
    ]),
}
list(models)

['DecisionTree', 'LogisticRegression', 'KNN']

## 4. Comparación inicial con validación cruzada

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
for name, pipe in models.items():
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring="accuracy", return_train_score=True)
    rows.append({
        "model": name,
        "train_acc": res["train_score"].mean(),
        "val_acc": res["test_score"].mean(),
        "val_std": res["test_score"].std(),
    })

initial_comparison = pd.DataFrame(rows).set_index("model").round(4)
initial_comparison["gap"] = (initial_comparison["train_acc"] - initial_comparison["val_acc"]).round(4)
initial_comparison

,train_acc,val_acc,val_std,gap
model,,,,
DecisionTree,1.0000,0.9022,0.0586,0.0978
LogisticRegression,1.0000,0.9791,0.0277,0.0209
KNN,0.9771,0.9507,0.0353,0.0264


## 5. Optimización de hiperparámetros

In [7]:
param_grids = {
    "DecisionTree": {
        "clf__max_depth": [2, 3, 4, 5, None],
        "clf__min_samples_leaf": [1, 3, 5],
    },
    "LogisticRegression": {
        "clf__C": [0.01, 0.1, 1, 10, 100],
    },
    "KNN": {
        "clf__n_neighbors": [3, 5, 7, 9, 11],
        "clf__weights": ["uniform", "distance"],
    },
}

searches = {}
rows = []
for name, pipe in models.items():
    gs = GridSearchCV(pipe, param_grids[name], scoring="accuracy", cv=cv, return_train_score=True, n_jobs=1)
    gs.fit(X_train, y_train)
    searches[name] = gs
    i = gs.best_index_
    rows.append({
        "model": name,
        "best_cv_acc": gs.best_score_,
        "cv_std": gs.cv_results_["std_test_score"][i],
        "train_acc": gs.cv_results_["mean_train_score"][i],
        "best_params": gs.best_params_,
    })

tuned_comparison = pd.DataFrame(rows).set_index("model")
tuned_comparison["gap"] = tuned_comparison["train_acc"] - tuned_comparison["best_cv_acc"]
tuned_comparison.round({"best_cv_acc": 4, "cv_std": 4, "train_acc": 4, "gap": 4})

,best_cv_acc,cv_std,train_acc,best_params,gap
model,,,,,
DecisionTree,0.9022,0.0586,0.9982,"{'clf__max_depth': 4, 'clf__min_samples_leaf': 1}",0.0960
LogisticRegression,0.9791,0.0277,1.0000,{'clf__C': 1},0.0209
KNN,0.9579,0.0405,0.9648,"{'clf__n_neighbors': 7, 'clf__weights': 'unifo...",0.0069


## 6. Selección y persistencia del campeón

In [8]:
champion_name = max(searches, key=lambda n: searches[n].best_score_)
champion = searches[champion_name].best_estimator_  # ya reajustado con todo train (refit=True)
print("Campeón:", champion_name, "| CV acc:", round(searches[champion_name].best_score_, 4))

joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

metadata = {
    "champion": champion_name,
    "best_params": searches[champion_name].best_params_,
    "cv_accuracy": float(searches[champion_name].best_score_),
    "cv_std": float(tuned_comparison.loc[champion_name, "cv_std"]),
    "cv_folds": 5,
    "scoring": "accuracy",
    "random_state": RANDOM_STATE,
    "features": feature_columns,
    "n_train": int(len(X_train)),
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(json.dumps(metadata, indent=2, default=str), encoding="utf-8")

cv_results = (tuned_comparison.reset_index()
              .assign(best_params=lambda d: d["best_params"].astype(str))
              .round(4))
cv_results.to_csv(REPORTS_DIR / "cv_results.csv", index=False)
cv_results

Campeón: LogisticRegression | CV acc: 0.9791


,model,best_cv_acc,cv_std,train_acc,best_params,gap
0,DecisionTree,0.9022,0.0586,0.9982,"{'clf__max_depth': 4, 'clf__min_samples_leaf': 1}",0.0960
1,LogisticRegression,0.9791,0.0277,1.0000,{'clf__C': 1},0.0209
2,KNN,0.9579,0.0405,0.9648,"{'clf__n_neighbors': 7, 'clf__weights': 'unifo...",0.0069


In [9]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

> Se eligió la regresión logística (C=1) por tener el mayor accuracy de CV (0.9791) y además la menor variabilidad entre folds (desviación 0.028, frente a 0.041 de KNN y 0.059 del árbol). El árbol de decisión es el peor: aunque en train llega a ~1.0, su validación es 0.902, una brecha de ~0.10 que indica sobreajuste y alta varianza con solo 142 filas. La regresión logística también alcanza 1.0 en train, con brecha de 0.021, pequeña y esperable con 13 variables y pocas muestras; KNN tiene la menor brecha (0.007) pero un desempeño de validación inferior (0.958). En complejidad, la regresión logística es un modelo lineal con pocos parámetros, rápido de entrenar y de servir, y es la más interpretable porque sus coeficientes por clase indican qué variables empujan cada predicción (coherente con el EDA, donde `flavanoids` y `proline` separan las clases). La diferencia con KNN es de ~2 puntos con desviaciones estándar comparables, así que no es abrumadora, pero la combinación de mejor media, menor varianza e interpretabilidad desempata a favor de la regresión logística.

## Checklist

- [x] Todo el procesamiento está dentro de pipelines.
- [x] Comparé tres familias de modelos.
- [x] Optimicé hiperparámetros con CV.
- [x] Elegí el campeón sin usar test.
- [x] Guardé el pipeline completo y sus metadatos.